# W2: 상품카피 자동화

상품명, 가격, 핵심 재료 정보를 기반으로 배달앱용 짧은 설명(80자 내외)과 키워드를 자동 생성합니다.


## Step 0: pip + gemini-2.0-flash + fm.findSystemFonts()+addfont()
```bash
!pip -q install google-generativeai pandas
```


In [ ]:
import io
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for font_path in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(font_path)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}

    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        out = model.generate_content(prompt)
        return getattr(out, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

products = [
    {"메뉴명": "치즈불닭면", "가격": 9800, "주요재료": "면,치즈,불닭소스"},
    {"메뉴명": "허니콤보", "가격": 13500, "주요재료": "닭다리살,꿀소스,파슬리"},
    {"메뉴명": "트러플카레라이스", "가격": 12000, "주요재료": "카레,쌀,트러플오일"}
]

def make_copy(row):
    prompt = f"상품명:{row['메뉴명']}, 가격:{row['가격']}, 주요재료:{row['주요재료']}. 배달앱에 들어갈 80자 이내 설명과 검색 키워드 5개 생성"
    fallback = f"[{row['메뉴명']}] {row['가격']}원, {row['주요재료']}로 만든 간편 배달 메뉴. 푸짐함과 빠른 배송"
    return use_gemini(prompt, fallback)

rows = []
for p in products:
    text = make_copy(p)
    rows.append({"메뉴명": p["메뉴명"], "설명": text})

out = pd.DataFrame(rows)
print(out)
